In [ ]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import tensorflow as tf
#nic
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix,roc_curve,roc_auc_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from keras import utils, callbacks
from tensorflow.keras import utils
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from keras.losses import CategoricalCrossentropy
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn import metrics
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import load_img, img_to_array, array_to_img
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from glob import glob
import sklearn


c:\Users\aless\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.utils import to_categorical
import os

# Initialize MediaPipe Hand detector
mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)


def create_combined_heatmap(image_rgb, target_size=64, sigma=1.5):
    """
    Create a single heatmap combining all 21 hand keypoints
    
    Args:
        image_rgb: RGB image for MediaPipe
        target_size: output size (e.g., 64)
        sigma: Gaussian spread (DEFAULT 1.5 for clearer keypoints)
    
    Returns:
        combined_heatmap: (target_size, target_size) single channel with all keypoints
        None: if no hand keypoints detected
    """
    results = hands_detector.process(image_rgb)
    
    # ⚠️ RETURN None IF NO KEYPOINTS DETECTED
    if not results.multi_hand_landmarks:
        return None
    
    # Initialize empty heatmap
    heatmap = np.zeros((target_size, target_size), dtype='float32')
    
    hand_landmarks = results.multi_hand_landmarks[0]
    
    # Create coordinate grids once
    x_grid, y_grid = np.meshgrid(np.arange(target_size), np.arange(target_size))
    
    for landmark in hand_landmarks.landmark:
        # Convert normalized coordinates to pixels
        x_px = int(landmark.x * target_size)
        y_px = int(landmark.y * target_size)
        
        # Clamp coordinates to valid range
        x_px = np.clip(x_px, 0, target_size - 1)
        y_px = np.clip(y_px, 0, target_size - 1)
        
        # Add Gaussian blob for this keypoint
        gaussian = np.exp(-((x_grid - x_px)**2 + (y_grid - y_px)**2) / (2 * sigma**2))
        heatmap = np.maximum(heatmap, gaussian)  # Take max to avoid overlapping
    
    return heatmap


def edge_detection(image):
    """Apply adaptive thresholding to enhance hand edges"""
    minValue = 70
    blur = cv2.GaussianBlur(image, (5, 5), 2)
    th3 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                 cv2.THRESH_BINARY_INV, 11, 2)
    ret, res = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return res


# ============================================================================
# OPTION A: RGB (3 channels) + HEATMAP (1 channel) = 4 CHANNELS TOTAL
# ============================================================================

def preprocess_image_rgb_heatmap(image_path, target_size=64):
    """
    RGB + Heatmap preprocessing → (64, 64, 4)
    
    Output channels:
    - Channels 0-2: RGB image (no edge detection)
    - Channel 3: Combined keypoint heatmap
    
    Args:
        image_path: path to image
        target_size: target dimension
    
    Returns:
        image with shape (64, 64, 4) OR None if no keypoints detected
    """
    try:
        # Load RGB
        img_rgb = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img_rgb is None:
            return None
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        # Resize with aspect ratio preservation
        h, w = img_rgb.shape[:2]
        aspect_ratio = h / w
        
        if aspect_ratio > 1:
            new_w = int(target_size / aspect_ratio)
            img_resized = cv2.resize(img_rgb, (new_w, target_size))
        else:
            new_h = int(target_size * aspect_ratio)
            img_resized = cv2.resize(img_rgb, (target_size, new_h))
        
        # Create square canvas
        canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        h_resized, w_resized = img_resized.shape[:2]
        y_offset = (target_size - h_resized) // 2
        x_offset = (target_size - w_resized) // 2
        canvas[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = img_resized
        
        # Extract combined heatmap
        heatmap = create_combined_heatmap(canvas, target_size)
        
        # ⚠️ RETURN None IF NO KEYPOINTS (excludes image from dataset)
        if heatmap is None:
            return None
        
        # Normalize RGB to [0, 1]
        rgb_normalized = canvas.astype('float32') / 255.0
        
        heatmap = np.expand_dims(heatmap, axis=-1)  # (64, 64, 1)
        
        # Stack: [R, G, B, Heatmap]
        stacked = np.concatenate([rgb_normalized, heatmap], axis=-1)  # (64, 64, 4)
        
        return stacked
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


# ============================================================================
# OPTION B: GRAYSCALE (1 channel) + HEATMAP (1 channel) = 2 CHANNELS TOTAL
# ============================================================================

def preprocess_image_gray_heatmap(image_path, target_size=64):
    """
    Grayscale + Heatmap preprocessing → (64, 64, 2)
    
    Output channels:
    - Channel 0: Edge-detected grayscale
    - Channel 1: Combined keypoint heatmap
    
    Args:
        image_path: path to image
        target_size: target dimension
    
    Returns:
        image with shape (64, 64, 2) OR None if no keypoints detected
    """
    try:
        # Load grayscale for edge detection
        img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None:
            return None
        
        # Load RGB for MediaPipe
        img_rgb = cv2.imread(image_path, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        # Apply edge detection
        img_edges = edge_detection(img_gray)
        
        # Resize with aspect ratio preservation
        h, w = img_edges.shape
        aspect_ratio = h / w
        
        if aspect_ratio > 1:
            new_w = int(target_size / aspect_ratio)
            gray_resized = cv2.resize(img_edges, (new_w, target_size))
            rgb_resized = cv2.resize(img_rgb, (new_w, target_size))
        else:
            new_h = int(target_size * aspect_ratio)
            gray_resized = cv2.resize(img_edges, (target_size, new_h))
            rgb_resized = cv2.resize(img_rgb, (target_size, new_h))
        
        # Create square canvas for grayscale
        canvas_gray = np.zeros((target_size, target_size), dtype=np.uint8)
        h_resized, w_resized = gray_resized.shape
        y_offset = (target_size - h_resized) // 2
        x_offset = (target_size - w_resized) // 2
        canvas_gray[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = gray_resized
        
        # Create canvas for RGB (for keypoint extraction)
        canvas_rgb = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        canvas_rgb[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = rgb_resized
        
        # Extract combined heatmap
        heatmap = create_combined_heatmap(canvas_rgb, target_size)
        
        # ⚠️ RETURN None IF NO KEYPOINTS (excludes image from dataset)
        if heatmap is None:
            return None
        
        # Normalize grayscale to [0, 1]
        gray_normalized = canvas_gray.astype('float32') / 255.0
        gray_channel = np.expand_dims(gray_normalized, axis=-1)  # (64, 64, 1)
        
        heatmap_channel = np.expand_dims(heatmap, axis=-1)  # (64, 64, 1)
        
        # Stack: [Grayscale, Heatmap]
        stacked = np.concatenate([gray_channel, heatmap_channel], axis=-1)  # (64, 64, 2)
        
        return stacked
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


# ============================================================================
# LOADER FUNCTION - CHOOSE YOUR OPTION
# ============================================================================

def load_images_from_folders_with_heatmap(train_path, test_path, target_size=64, mode='grayscale'):
    """
    Load images with heatmaps
    
    Args:
        train_path: training folder path
        test_path: test folder path
        target_size: image size (default 64)
        mode: 'grayscale' for 2 channels or 'rgb' for 4 channels
    
    Returns:
        x_train, x_test with shape (..., 64, 64, 2) or (..., 64, 64, 4)
        y_train, y_test with one-hot encoding
    """
    x_train = []
    y_train = []
    x_test = []
    y_test = []
    
    # Counters for excluded images
    excluded_train = 0
    excluded_test = 0
    
    # Choose preprocessing function
    if mode == 'rgb':
        preprocess_fn = preprocess_image_rgb_heatmap
        expected_channels = 4
        print("🎨 Mode: RGB + Heatmap (4 channels)")
    else:  # grayscale
        preprocess_fn = preprocess_image_gray_heatmap
        expected_channels = 2
        print("⚫ Mode: Grayscale + Heatmap (2 channels)")
    
    character_folders = sorted([d for d in os.listdir(train_path) 
                                if os.path.isdir(os.path.join(train_path, d))])
    
    print(f"Found {len(character_folders)} character classes: {character_folders}")
    print("⚠️  Images without detected keypoints will be EXCLUDED from dataset")
    print("📍 Using sigma=1.5 for precise keypoint localization")
    print("\n" + "="*80)
    
    for class_idx, char_folder in enumerate(character_folders):
        train_char_path = os.path.join(train_path, char_folder)
        test_char_path = os.path.join(test_path, char_folder)
        
        # Load training images
        train_count = 0
        train_excluded = 0
        if os.path.exists(train_char_path):
            for img_file in os.listdir(train_char_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(train_char_path, img_file)
                    img = preprocess_fn(img_path, target_size)
                    if img is not None and img.shape == (target_size, target_size, expected_channels):
                        x_train.append(img)
                        y_train.append(class_idx)
                        train_count += 1
                    elif img is None:
                        train_excluded += 1
        
        # Load test images
        test_count = 0
        test_excluded = 0
        if os.path.exists(test_char_path):
            for img_file in os.listdir(test_char_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(test_char_path, img_file)
                    img = preprocess_fn(img_path, target_size)
                    if img is not None and img.shape == (target_size, target_size, expected_channels):
                        x_test.append(img)
                        y_test.append(class_idx)
                        test_count += 1
                    elif img is None:
                        test_excluded += 1
        
        excluded_train += train_excluded
        excluded_test += test_excluded
        
        status = ""
        if train_excluded > 0 or test_excluded > 0:
            status = f" | ❌ Excluded: Train={train_excluded}, Test={test_excluded}"
        
        print(f"Class {class_idx:2d} ({char_folder}): Train={train_count:4d} | Test={test_count:4d}{status}")
    
    print("="*80)
    
    if len(x_train) == 0 or len(x_test) == 0:
        raise ValueError("No images found! Check folder paths.")
    
    x_train = np.array(x_train)
    x_test = np.array(x_test)
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    print(f"\n✓ Training set: {x_train.shape} images")
    print(f"✓ Test set: {x_test.shape} images")
    print(f"✓ Classes: {len(np.unique(y_train))}")
    
    if excluded_train > 0 or excluded_test > 0:
        print(f"\n⚠️  Total excluded (no keypoints): Train={excluded_train}, Test={excluded_test}")
    
    num_classes = len(character_folders)
    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)
    
    print(f"✓ Labels one-hot encoded: {y_train.shape}")
    
    return x_train, x_test, y_train, y_test


# ============================================================================

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# Load from local folders instead of kagglehub
import os

# Update these paths to your local folder locations
train_folder = "asl_alphabet_train"  # or full path if needed
test_folder = "asl_alphabet_test"

# Check if folders exist
if not os.path.exists(train_folder):
    print(f"⚠️ Folder not found: {train_folder}")
    print(f"Current directory: {os.getcwd()}")
    print(f"Available items: {os.listdir('.')}")
else:
    print(f"✓ Found training folder: {train_folder}")
    print(f"✓ Found test folder: {test_folder}")

x_train, x_test, y_train, y_test = load_images_from_folders_with_heatmap(train_folder, test_folder, target_size=64, mode="rgb")


✓ Found training folder: asl_alphabet_train
✓ Found test folder: asl_alphabet_test
🎨 Mode: RGB + Heatmap (4 channels)
Found 26 character classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
⚠️  Images without detected keypoints will be EXCLUDED from dataset
📍 Using sigma=1.5 for precise keypoint localization

Class  0 (A): Train=1172 | Test= 115 | ❌ Excluded: Train=223, Test=0
Class  1 (B): Train=1139 | Test= 115 | ❌ Excluded: Train=256, Test=0
Class  2 (C): Train=1228 | Test=  97 | ❌ Excluded: Train=347, Test=18
Class  3 (D): Train=1284 | Test= 115 | ❌ Excluded: Train=159, Test=0
Class  4 (E): Train=1220 | Test= 115 | ❌ Excluded: Train=189, Test=0
Class  5 (F): Train=1293 | Test= 115 | ❌ Excluded: Train=79, Test=0
Class  6 (G): Train=1221 | Test= 104 | ❌ Excluded: Train=161, Test=11
Class  7 (H): Train=1267 | Test= 115 | ❌ Excluded: Train=128, Test=0
Class  8 (I): Train=1230 | Test= 115 | ❌ Excluded

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
from collections import Counter

print("="*70)
print("SPLIT CONFORMAL PREDICTION WITH SOFTMAX FOR CNN")
print("="*70)

# ===============================
# 0) VERIFY AND PREPARE TEST DATA
# ===============================
print("\n0. Verifying already-loaded test data...")
X_test = x_test  # Use the already loaded data

# Handle different label formats
if len(y_test.shape) > 1:
    print(f"   Original labels shape: {y_test.shape}")
    if y_test.shape[1] > 1:  # One-hot encoded
        print("   Converting one-hot labels to class indices...")
        y_test_flat = np.argmax(y_test, axis=1)
    else:  # Column vector
        y_test_flat = y_test.flatten()
else:
    y_test_flat = y_test

print(f"   Test images shape: {X_test.shape}")
print(f"   Test labels shape: {y_test_flat.shape}")
print(f"   Image shape: {X_test.shape[1:]}")
print(f"   Pixel value range: [{X_test.min():.2f}, {X_test.max():.2f}]")
print(f"   Unique labels: {sorted(np.unique(y_test_flat))}")
print(f"   Number of classes: {len(np.unique(y_test_flat))}")

# ===============================
# 1) LOAD THE CNN MODEL
# ===============================
print("\n1. Loading CNN model...")
model = keras.models.load_model("ASL_predictormemecambioheatmap.keras")
print(f"   Model loaded successfully!")
print(f"   Model architecture: {model.name}")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

# ===============================
# 2) EXTRACT SOFTMAX PROBABILITIES
# ===============================
print("\n2. Extracting softmax probabilities...")

# Check if model already outputs softmax
# If the last layer is softmax, we're good; otherwise we need to add it
last_layer = model.layers[-1]
print(f"   Last layer: {last_layer.name} ({last_layer.__class__.__name__})")

# Get softmax predictions function
def get_softmax_predictions(X):
    """Get softmax probabilities from the model"""
    predictions = model.predict(X, verbose=0)
    # Ensure probabilities sum to 1 (in case of numerical issues)
    predictions = predictions / predictions.sum(axis=1, keepdims=True)
    return predictions

# Test on a small batch
print(f"   Testing softmax extraction on sample data...")
sample_probs = get_softmax_predictions(X_test[:5])
print(f"   Sample probabilities shape: {sample_probs.shape}")
print(f"   Probabilities sum to 1: {np.allclose(sample_probs.sum(axis=1), 1.0)}")

# ===============================
# 3) SPLIT TEST SET
# ===============================
print("\n3. Splitting test set into calibration and final test...")
X_calib, X_test_final, y_calib, y_test_final = train_test_split(
    X_test, y_test_flat, test_size=0.5, random_state=42, stratify=y_test_flat
)
print(f"   Calibration set: {X_calib.shape[0]} samples")
print(f"   Final test set: {X_test_final.shape[0]} samples")

# ===============================
# 4) COMPUTE NONCONFORMITY SCORES ON CALIBRATION SET
# ===============================
print("\n4. Computing nonconformity scores on calibration set...")
print(f"   Nonconformity score = 1 - softmax(true_class)")

# Get softmax predictions for calibration set
softmax_calib = get_softmax_predictions(X_calib)

# Get list of classes (assuming labels are already indices 0, 1, 2, ...)
classes_list = sorted(np.unique(y_test_flat))
n_classes = len(classes_list)
print(f"   Number of classes: {n_classes}")

# Compute nonconformity scores: 1 - p(true class)
nonconformity_scores_calib = []
for i in range(len(y_calib)):
    true_class_idx = int(y_calib[i])  # Already an index
    prob_true_class = softmax_calib[i, true_class_idx]
    nonconf_score = 1 - prob_true_class
    nonconformity_scores_calib.append(nonconf_score)

nonconformity_scores_calib = np.array(nonconformity_scores_calib)
print(f"   Nonconformity scores computed: {len(nonconformity_scores_calib)}")
print(f"   Score range: [{nonconformity_scores_calib.min():.4f}, {nonconformity_scores_calib.max():.4f}]")
print(f"   Mean score: {nonconformity_scores_calib.mean():.4f}")

# ===============================
# 5) COMPUTE QUANTILE (α-level)
# ===============================
alpha = 0.1  # Miscoverage rate (target: 90% coverage)
print(f"\n5. Computing quantile for α = {alpha} (target coverage: {(1-alpha)*100:.0f}%)")

n = len(nonconformity_scores_calib)
# Conformal quantile formula: ceil((n+1)(1-α))/n
q_level = np.ceil((n + 1) * (1 - alpha)) / n
q_level = min(q_level, 1.0)  # Cap at 1.0
quantile = np.quantile(nonconformity_scores_calib, q_level)

print(f"   Calibration set size (n): {n}")
print(f"   Quantile level: {q_level:.4f}")
print(f"   Quantile threshold: {quantile:.4f}")

# ===============================
# 6) BUILD PREDICTION SETS ON FINAL TEST SET
# ===============================
print("\n6. Building prediction sets on final test set...")

# Get softmax predictions for final test set
softmax_test = get_softmax_predictions(X_test_final)

def build_prediction_sets(softmax_probs, quantile, n_classes):
    """
    Build prediction sets using Split Conformal Prediction
    Include all classes where 1 - p(class) ≤ quantile
    """
    prediction_sets = []
    point_predictions = []
    
    for probs in softmax_probs:
        pred_set = []
        
        # Check each class
        for class_idx in range(n_classes):
            nonconf_score = 1 - probs[class_idx]
            if nonconf_score <= quantile:
                pred_set.append(class_idx)
        
        # Handle edge case: empty set (shouldn't happen with proper quantile)
        if len(pred_set) == 0:
            # Include the class with highest probability
            best_idx = np.argmax(probs)
            pred_set = [best_idx]
        
        prediction_sets.append(pred_set)
        
        # Point prediction: argmax
        point_pred_idx = np.argmax(probs)
        point_predictions.append(point_pred_idx)
    
    return point_predictions, prediction_sets

y_pred_conf, pred_sets = build_prediction_sets(softmax_test, quantile, n_classes)
y_pred_conf = np.array(y_pred_conf)

print(f"   Prediction sets built for {len(pred_sets)} samples")

# ===============================
# 7) EVALUATION
# ===============================
print("\n" + "="*70)
print(f"SPLIT CONFORMAL PREDICTION RESULTS (α = {alpha})")
print("="*70)

# Accuracy (point prediction)
accuracy = accuracy_score(y_test_final, y_pred_conf)
print(f"\n📊 1. ACCURACY (Point Prediction): {accuracy*100:.2f}%")

# Coverage (proportion of times true class is in prediction set)
coverage_list = [1 if y_test_final[i] in pred_sets[i] else 0 
                 for i in range(len(y_test_final))]
avg_coverage = np.mean(coverage_list)
coverage_gap = (avg_coverage - (1 - alpha)) * 100

print(f"\n🎯 2. COVERAGE:")
print(f"   Average Coverage: {avg_coverage*100:.2f}%")
print(f"   Target Coverage: {(1-alpha)*100:.0f}%")
print(f"   Coverage Gap: {coverage_gap:+.2f}%")
if avg_coverage >= (1 - alpha):
    print(f"   ✓ Coverage guarantee satisfied!")
else:
    print(f"   ⚠ Coverage below target (may be due to small calibration set)")

# Cardinality (size of prediction sets)
cardinalities = [len(pred_set) for pred_set in pred_sets]
avg_cardinality = np.mean(cardinalities)
efficiency = (1 - (avg_cardinality - 1) / (n_classes - 1)) * 100

print(f"\n📦 3. CARDINALITY:")
print(f"   Average Set Size: {avg_cardinality:.2f}")
print(f"   Min Set Size: {min(cardinalities)}")
print(f"   Max Set Size: {max(cardinalities)}")
print(f"   Median Set Size: {np.median(cardinalities):.1f}")
print(f"   Efficiency: {efficiency:.1f}% (lower avg size = better)")

# Per-class metrics
print(f"\n📋 4. PER-CLASS METRICS:")
print(f"   {'Class':<6} {'Coverage':<10} {'Avg Size':<10} {'Samples':<8}")
print(f"   {'-'*40}")
for cls_idx in classes_list:
    mask = y_test_final == cls_idx
    n_samples = np.sum(mask)
    if n_samples > 0:
        cls_coverage = np.mean([1 if cls_idx in pred_sets[i] else 0 
                                for i in range(len(y_test_final)) if mask[i]])
        cls_cardinality = np.mean([len(pred_sets[i]) 
                                   for i in range(len(y_test_final)) if mask[i]])
        print(f"   {cls_idx:<6} {cls_coverage*100:>6.1f}%    {cls_cardinality:>6.2f}     {n_samples:<8}")

# ===============================
# 8) DETAILED EXAMPLES
# ===============================
print("\n" + "="*70)
print("EXAMPLES OF PREDICTION SETS (First 50 samples)")
print("="*70)
print(f"{'#':<4} {'True':<5} {'Pred':<5} {'✓':<2} {'Prediction Set':<40} {'Size':<5} {'Covered':<8}")
print("-"*80)

n_examples = min(50, len(X_test_final))
for i in range(n_examples):
    true_class = int(y_test_final[i])
    point_pred = int(y_pred_conf[i])
    pred_set = [int(x) for x in pred_sets[i]]
    correct = "✓" if point_pred == true_class else "✗"
    in_set = "✓" if true_class in pred_set else "✗"
    
    # Format prediction set (truncate if too long)
    set_str = str(sorted(pred_set))
    if len(set_str) > 38:
        set_str = set_str[:35] + "..."
    
    print(f"{i+1:<4} {true_class:<5} {point_pred:<5} {correct:<2} {set_str:<40} {len(pred_set):<5} {in_set:<8}")

# ===============================
# 9) DISTRIBUTION OF SET SIZES
# ===============================
print("\n" + "="*70)
print("DISTRIBUTION OF PREDICTION SET SIZES")
print("="*70)
size_dist = Counter(cardinalities)
max_count = max(size_dist.values())
for size in sorted(size_dist.keys()):
    count = size_dist[size]
    pct = count / len(cardinalities) * 100
    bar_length = int((count / max_count) * 40)
    bar = "█" * bar_length
    print(f"Size {size:2d}: {count:4d} samples ({pct:5.1f}%) {bar}")

# ===============================
# 10) CONFIDENCE ANALYSIS
# ===============================
print("\n" + "="*70)
print("CONFIDENCE ANALYSIS")
print("="*70)

# Analyze relationship between max probability and set size
max_probs = np.max(softmax_test, axis=1)
print(f"\nMax Softmax Probability Statistics:")
print(f"   Mean: {max_probs.mean():.4f}")
print(f"   Std:  {max_probs.std():.4f}")
print(f"   Min:  {max_probs.min():.4f}")
print(f"   Max:  {max_probs.max():.4f}")

# Correlation between confidence and set size
corr, p_value = spearmanr(max_probs, cardinalities)
print(f"\nCorrelation between max probability and set size:")
print(f"   Spearman ρ = {corr:.3f} (p={p_value:.4f})")
if corr < -0.5:
    print(f"   ✓ Strong negative correlation: higher confidence → smaller sets")

# ===============================
# 11) SUMMARY
# ===============================
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"✓ Point Prediction Accuracy: {accuracy*100:.2f}%")
print(f"✓ Average Coverage: {avg_coverage*100:.2f}% (Target: {(1-alpha)*100:.0f}%)")
print(f"✓ Average Set Size: {avg_cardinality:.2f} classes")
print(f"✓ Model Efficiency: {efficiency:.1f}%")
print(f"✓ Calibration set: {len(X_calib)} samples")
print(f"✓ Test set: {len(X_test_final)} samples")
print("="*70)

SPLIT CONFORMAL PREDICTION WITH SOFTMAX FOR CNN

0. Verifying already-loaded test data...
   Original labels shape: (2925, 26)
   Converting one-hot labels to class indices...
   Test images shape: (2925, 64, 64, 4)
   Test labels shape: (2925,)
   Image shape: (64, 64, 4)
   Pixel value range: [0.00, 1.00]
   Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]
   Number of classes: 26

1. Loading CNN model...
   Model loaded successfully!
   Model architecture: sequential_12
   Input shape: (None, 64, 64, 4)
   Output shape: (None, 26)

2. Extracting softmax probabilities...
   Last layer: dense_28 (Dense)
   Testing softmax extraction on sample data...
   Sample probabili